In [36]:
from langgraph.graph import StateGraph , START , END
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from typing import TypedDict
from dotenv import load_dotenv
import os

In [37]:
load_dotenv()

True

In [38]:
llm = HuggingFaceEndpoint(
    repo_id="deepseek-ai/DeepSeek-V4-Pro",
    task="text-generation",
    huggingfacehub_api_token=os.getenv("HUGGINGFACEHUB_ACCESS_TOKEN"),
    temperature=0.8
)
model = ChatHuggingFace(llm = llm)

In [39]:
class blogstate(TypedDict):
    title : str
    outline :str
    content : str
    answer : str

In [40]:
def createoutline(state:blogstate) -> blogstate:
    title = state["title"]
    prompt = f'Generate an outline for the topic{title}'

    outline = model.invoke(prompt)

    state['outline'] = outline
    return state

In [41]:
def blog(state:blogstate) -> blogstate:
    title = state['title']
    outline = state['outline']

    prompt = f'write a detailed blog on the title{title} , using following outline\n{outline}'

    content = model.invoke(prompt).content
    state['content'] = content
    return state

In [42]:
def rate(state:blogstate) -> blogstate:
    title = state['title']
    outline = state['outline']
    content = state['content']

    prompt = f'Based on the given outline : {outline} , rate this blog out of 10 : {content}'

    answer = model.invoke(prompt)
    state['answer'] = answer
    return state

In [43]:
graph = StateGraph(blogstate)

graph.add_node('create outline', createoutline)
graph.add_node('create blog', blog)
graph.add_node('rate the  blog', rate)

graph.add_edge(START,'create outline')
graph.add_edge('create outline','create blog')
graph.add_edge('create blog', 'rate the  blog')
graph.add_edge('rate the  blog', END)

workflow = graph.compile()

In [44]:
intial = {'title' : ' Rise of AI jobs in India '}

final = workflow.invoke(intial)

print(final['content'])

Here is a comprehensive blog post based on your outline.

---

# The New Frontier: A Roadmap to India’s AI-Powered Job Market

**By [Your Name/Company Name]**

---

### Executive Summary

India is not just participating in the global Artificial Intelligence (AI) revolution; it is architecting it. Long celebrated as the world’s back-office, the country has rapidly transformed into a "brain-office"—a primary hub for AI research, development, and deployment. With a talent pool exceeding 2 million STEM graduates annually and a digital-first policy environment, India’s AI job market is projected to grow at a compound annual growth rate (CAGR) exceeding 30-40%, reaching a market size of several billion dollars by 2027.

The core premise driving this boom is a fundamental structural shift: the narrative is moving from providing low-cost IT services to creating high-value AI products and platforms. This shift is fueled by a rare convergence of factors—an explosion of Global Capability Centers 